In [ ]:
# Import what you need here

import requests
import keyring
from dotenv import load_dotenv
load_dotenv()
import os
import pandas as pd


In [ ]:

API_key=os.getenv("API_key_.gov")

Some info from the documentation I may want to refer back to:

Base URL for specific congress and chamber: https://api.congress.gov/v3/committee-meeting/[118]/[house]?api_key=[API_key]

<title> (e.g., Legislative hearing on: •H.R. 1246 (Rep. Hageman), To authorize leases of up to 99 years for land held in trust for federally recognized Indian tribes; and•H.R. 1532 (Rep. Hageman), To authorize any Indian Tribe to lease, sell, convey, warrant, or otherwise transfer real property to which that Indian Tribe holds fee title without the consent of the Federal Government, and for other purposes.)

<witnesses>
    Container for witnesses associated with the meeting. A <witnesses> element may include the following children:
    <item>
    Container for a witness associated with the meeting. An <item> element may include the following children:
    <name> (e.g., Mr. Thomas DiNanno)
    The name of the witness.
    <position> (e.g., Assistant Administrator, Grant Programs Directorate, Federal Emergency Management Agency)
    The witness's professional position.
    <organization> (e.g., U.S. Department of Homeland Security)
    The witness's organization.

<committees> Container for the committees or subcommittees the held the meeting. A <committees> element may include the following children:
        <item>
        Container for an individual committee or subcommittee that held the meeting. An <item> element is repeatable and may include the following children:
        <systemCode> (e.g., hsii24)
        Unique ID value for the committee or subcommittee.
        <url> (e.g., https://api.congress.gov/v3/committee/house/hsii24)
        A referrer URL to the committee or subcommittee item in the API. Documentation for the committee API is available here.
        <name> (e.g., House Natural Resources Subcommittee on Indian and Insular Affairs)
        The name of the committee or subcommittee.


### TESTING DATA SECTION ###

Call for all committee meetings during the current congress (#119) in the House:

In [ ]:
# url = f'https://api.congress.gov/v3/committee-meeting/119/house?api_key={API_key}'

# request = requests.get(url)

# attempt = request.json()


Looking at the data

In [ ]:
# attempt.keys()

# len(attempt['committeeMeetings']) #This shows us that we're only getting 1 page with 20 results

Trying to increase limit so I don't have to go through so many pages.

In [ ]:
# url = f'https://api.congress.gov/v3/committee-meeting/119/house?limit=250&api_key={API_key}'

# request = requests.get(url)

# attempt = request.json()

# attempt.keys()

# attempt['committeeMeetings']

# len(attempt['committeeMeetings'])


It seems to give you the next url to run in attempt['pagination']['next']. Testing that here:

In [ ]:
# attempt['pagination']['next']

After trial and error, it seems 250/page is the limit.



### CSV BUILDING SECTION ###

I first built this section to call all meetings in the House in Congress 119.

I am commenting this section out because I then built a general section to call any Congress for either chamber.

In [ ]:
# house119_allmeetings = [] #create empty list to fill of all committee meetings

# url = f'https://api.congress.gov/v3/committee-meeting/119/house?limit=250&api_key={API_key}'

# #initial request:

# initial_request = requests.get(url)
# house119 = initial_request.json()

# house119_allmeetings = house119['committeeMeetings']


In [ ]:
# len(house119_allmeetings) # shows this has done 1 page of 250 items (of 6 pages, 1500 items)

In [ ]:
# # get total pagecount for this call, with 250/page

# page_count = house119['pagination']['count'] / 250
# page_count = int(page_count)

# # now run through the pages of results
# for i in range(1, page_count): # for all of the pages (it tells me the number of pages in the initial call)
#     new_url = f'https://api.congress.gov/v3/committee-meeting/119/house?offset={i*250}&limit=250&format=json&api_key={API_key}' # get the new url
#     house119_newbatch = requests.get(new_url).json()
    
#     house119_allmeetings.extend(house119_newbatch['committeeMeetings'])


In [ ]:
# len(house119_allmeetings)


Check that all the IDs for all the meetings are unique

In [ ]:
# nonduplicate_ids = []
# duplicate_ids = []

# for i in range(len(house119_allmeetings)):

#     id = house119_allmeetings[i]['eventId']

#     if id not in nonduplicate_ids:
#         nonduplicate_ids.append(id)
#     else:
#         duplicate_ids.append(id)

# print(len(nonduplicate_ids))
# print(len(duplicate_ids))


Woohoo finally!

Now filter for environment-related committees
Also, add the date the meeting actually occured to the entry in house119_enviromeetings.

Do this by looking at each committee name.

In [ ]:
# len(house119_allmeetings)

In [ ]:
# for i in range(len(house119_allmeetings)):


#     # open up the meeting URL via a new request
#     url = f'{house119_allmeetings[i]['url']}&api_key={API_key}'
#     meeting = requests.get(url).json()

#     # define new entries in the meeting item that I will need later.
#     # For specific committees, and then also for date, title of meeting, witnesses
#     committees = meeting['committeeMeeting']['committees']
#     house119_allmeetings[i]['committees'] = committees

#     date = meeting['committeeMeeting']['date']
#     house119_allmeetings[i]['date'] = date

#     title = meeting['committeeMeeting']['title']
#     house119_allmeetings[i]['title'] = title

#     witnesses = meeting['committeeMeeting'].get('witnesses', 'none') # if there is no witnesses entry, return none
#     house119_allmeetings[i]['witnesses'] = witnesses

#     print(house119_allmeetings[i])


Make a dataframe for all meetings

In [ ]:
# df_house_119_allmeetings = pd.DataFrame.from_dict(house119_allmeetings)

Now sort for just environment committee meetings

In [ ]:
# #list of House environment committees 
# environment_committees = ['House Energy and Commerce', 'Committee on Natural Resources', 'House Natural Resources']
# environment_committee_links = ['https://naturalresources.house.gov/', 'https://energycommerce.house.gov']

# # empty dataframe to fill with a subset of df_house_119_allmeetings
# df_house119_enviromeetings = pd.DataFrame()

# for index, committee_list in df_house_119_allmeetings["committees"].items():

#     print(index)    

#     for committee in committee_list:

#         if 'name' in committee: # If we are working with name entries for the committees, take a look at each of those

#             # Add the meeting to the list of environment meetings, if an item in the list of environment_committees list matches the start of the committee name,
#             # (I use starts with rather than = because they often include a subcommittee name after the committee name within the name entry)
            
#             if any(committee['name'].startswith(prefix) for prefix in environment_committees):
#                 df_house119_enviromeetings = pd.concat([df_house119_enviromeetings, df_house_119_allmeetings.iloc[[index]]], ignore_index=True)

#             else:
#                 pass

#                 print(f'I passed over {committee['name']}')

#         elif 'name' not in committee:

#             url = committee['url']
#             committee_url = f'{url}&api_key={API_key}'
#             missing_name_request = requests.get(committee_url).json()
#             committee_link = missing_name_request['committee']['committeeWebsiteUrl']
            
#             if committee_link in environment_committee_links:
#                 df_house119_enviromeetings = pd.concat([df_house119_enviromeetings, df_house_119_allmeetings.iloc[[index]]], ignore_index=True)
            
#             else:
#                 pass
#                 print(f'I passed over {committee_link}.')

In [ ]:
# df_house119_enviromeetings.head()


export the csv

In [ ]:
# df_house119_enviromeetings.to_csv('house119.csv', index=False)

General script for any Congress

In [ ]:
congress_number = 114

chamber = 'senate'

In [ ]:
all_meetings = [] #create empty list to fill of all committee meetings

url = f'https://api.congress.gov/v3/committee-meeting/{congress_number}/{chamber}?limit=250&api_key={API_key}'

#initial request:

initial_request = requests.get(url)
general = initial_request.json()

all_meetings = general['committeeMeetings']

all_meetings

In [ ]:
len(all_meetings)

Get page count

In [ ]:
# get total pagecount for this call, with 250/page

page_count = general['pagination']['count'] / 250
page_count = int(page_count)


Then begin creating full list of commitee meetings with first page of 250 results

In [ ]:

# now run through the pages of results
for i in range(1, page_count): # for all of the pages (it tells me the number of pages in the initial call)
    new_url = f'https://api.congress.gov/v3/committee-meeting/{congress_number}/{chamber}?offset={i*250}&limit=250&format=json&api_key={API_key}' # get the new url
    newbatch = requests.get(new_url).json()
    
    all_meetings.extend(newbatch['committeeMeetings'])

Check if it worked

In [ ]:
all_meetings

## Grab all meetings on remaining pages, running through each page. ##

In [ ]:
#calulating estimated run time. Approx. 0.66 - 0.93 seconds per meeting.

estimated_run_time = len(all_meetings) * .8

f'Estimated run time is {estimated_run_time / 60} minutes.'

In [ ]:
handcheck_list = []

for i in range(len(all_meetings)):

    # open up the meeting URL via a new request
    url = f'{all_meetings[i]['url']}&api_key={API_key}'
    meeting = requests.get(url).json()

    # define new entries in the meeting item that I will need later.
    # For specific committees, and then also for date, title of meeting, witnesses

    if 'committees' in meeting['committeeMeeting']: # a few of the entries randomly dont have any committees listed, so need to run this if/elif so it doesnt break in those cases

        committees = meeting['committeeMeeting']['committees']
        all_meetings[i]['committees'] = committees

    elif 'committees' not in meeting['committeeMeeting']:
        
        handcheck_list.append(meeting['committeeMeeting'])

    date = meeting['committeeMeeting']['date']
    all_meetings[i]['date'] = date

    title = meeting['committeeMeeting']['title']
    all_meetings[i]['title'] = title

    witnesses = meeting['committeeMeeting'].get('witnesses', 'none') # if there is no witnesses entry, return none
    all_meetings[i]['witnesses'] = witnesses


    print(all_meetings[i])
    

In [ ]:
handcheck_list

In [ ]:
print(all_meetings[1]["committees"])

## Make a dataframe with all the meetings ##

In [ ]:
df = pd.DataFrame.from_dict(all_meetings)

Ensure no duplicates

In [ ]:
df[df.duplicated(subset=['eventId'])]

If duplicates, delete them:

In [ ]:
df.drop_duplicates(subset=['eventId'], keep='first', inplace=True)

Checking that the above deletion worked:

In [ ]:
df[df.duplicated(subset=['eventId'])]

## Make a new dataframe with just the environment committee meetings. ##

Calculate estimated run time (varies substantially from 0.045 seconds/all_meetings entry to 0.1922)

In [ ]:
estimated_run_time = len(all_meetings) * .1922

f'Estimated run time is up to {estimated_run_time / 60} minutes.'

In [ ]:
list_to_handcheck = [] # make a list of items I'm struggling to categorize, for handchecking

df_enviro_meetings = pd.DataFrame() # empty dataframe to fill with a subset of all_meetings

count = 0 # for testing/checking

if chamber == 'house': # RUN IF HOUSE

    #list of House environment committees 

    house_environment_committees = ['House Energy and Commerce', 'Committee on Natural Resources', 'House Natural Resources']
    house_environment_committee_links = ['https://naturalresources.house.gov/', 'https://energycommerce.house.gov']

    enviro_indices = [] # make a blank list of rows (indices) that have an environment committee

    for index, committee_list in df["committees"].items():

        # print(index)    

        for committee in committee_list:

            # If there are name entries for the committees, take a look at each of those

            if 'name' in committee: 

                # Add the meeting to the list of environment meetings, if an item in the list of environment_committees list matches the start of the committee name,
                # (I use starts with rather than = because they often include a subcommittee name after the committee name within the name entry)
                
                if any(committee['name'].startswith(prefix) for prefix in house_environment_committees): # if any of the committees start with any of the titles in house_environment_committees list 
                    enviro_indices.append(index)

                    # just for visibility/testing 
                    print(f'{index}, {committee['name']} meeting added.')
                    count = count + 1

                    break # Stop checking other committees for this row; we found a match. Prevents duplicates.


                else: # otherwise pass over it
                    pass
                    # print(f'I passed over {committee['name']}')

            # If not, try the URLs. It's usually pretty easy to see from there what the committee is.

            elif 'name' not in committee:

                url = committee['url']
                committee_url = f'{url}&api_key={API_key}'
                missing_name_request = requests.get(committee_url).json()

                if 'committeeWebsiteUrl' in missing_name_request['committee']:
                    committee_link = missing_name_request['committee']['committeeWebsiteUrl']
                
                    if committee_link in house_environment_committee_links:
                        enviro_indices.append(index)
                        
                        # just for visibility/testing 
                        print(f'{index}, {committee_link} meeting added.')
                        count = count + 1
                        
                        break # Stop checking other committees for this row; we found a match. Prevents duplicates.
                
                    else:
                        pass
                        # print(f'I passed over {committee_link}.')

                else:
                    # In the rare cases that the URL doesn't exist either, were making a list of elements to handcheck.
                    # We'll add the historic name of the committee so we can tell whther environment related.
                    historic_name = missing_name_request['committee']['history'][0]['libraryOfCongressName']
                    list_to_handcheck.append(f'{index}, {historic_name}')


elif chamber == 'senate': # RUN IF SENATE

    print('running senate script')

    #list of Senate environment committees 

    senate_environment_committees = ['Senate Environment and Public Works', 'Environment and Public Works', 'Senate Energy and Natural Resources', 'Energy and Natural Resources']
    senate_environment_committee_links = ['https://www.epw.senate.gov/public/', 'https://www.energy.senate.gov']

    enviro_indices = [] # make a blank list of rows (indices) that have an environment committee

    for index, committee_list in df['committees'].items():

        # print(index) # for testing

        # Checking if committee_list is empty. What this does is temporarily drops na values and then check if its empty. I added this if/else at the top because it broke when there was a nan in committee list

        clean_series = pd.Series(committee_list).dropna()
        if clean_series.empty or (clean_series == 'nan').all(): 
            list_to_handcheck.append(df.iloc[index])
            print(f'No committee list found for {df['committees']}.')

        else:
    
            for committee in committee_list:

                # If there are name entries for the committees, take a look at each of those

                if 'name' in committee: 

                    # Add the meeting to the list of environment meetings, if an item in the list of environment_committees list matches the start of the committee name,
                    # (I use starts with rather than = because they often include a subcommittee name after the committee name within the name entry)
                    
                    if any(committee['name'].startswith(prefix) for prefix in senate_environment_committees): # if any of the committees start with any of the titles in house_environment_committees list 
                        enviro_indices.append(index)

                        # just for visibility/testing 
                        print(f'{index}, {committee['name']} meeting added.')
                        count = count + 1

                        break # Stop checking other committees for this row; we found a match. Prevents duplicates.


                    else: # otherwise pass over it
                        pass
                        print(f'I passed over {committee['name']}')

                # If not, try the URLs. It's usually pretty easy to see from there what the committee is.

                elif 'name' not in committee:

                    url = committee['url']
                    committee_url = f'{url}&api_key={API_key}'
                    missing_name_request = requests.get(committee_url).json()

                    if 'committeeWebsiteUrl' in missing_name_request['committee']:
                        committee_link = missing_name_request['committee']['committeeWebsiteUrl']
                    
                        if committee_link in senate_environment_committee_links:
                            enviro_indices.append(index)
                            
                            # just for visibility/testing 
                            print(f'{index}, {committee_link} meeting added.')
                            count = count + 1
                            
                            break # Stop checking other committees for this row; we found a match. Prevents duplicates.
                    
                        else:
                            pass
                            # print(f'I passed over {committee_link}.')

                    else:
                        # In the rare cases that the URL doesn't exist either, were making a list of elements to handcheck.
                        # We'll add the historic name of the committee so we can tell whther environment related.
                        historic_name = missing_name_request['committee']['history'][0]['libraryOfCongressName']
                        list_to_handcheck.append(f'{index}, {historic_name}')

print(count) #print the number that should be in the df just to make sure its working

# Create the final filtered DataFrame all at once

df_enviro_meetings = df.loc[enviro_indices].copy()


Look through list of entries to hand check.

In [ ]:
list_to_handcheck

In [ ]:
df_enviro_meetings

Export as CSV

In [ ]:
df.to_csv(f'{chamber}{congress_number}_allmeetings.csv')
df_enviro_meetings.to_csv(f'{chamber}{congress_number}_enviromeetings.csv')